# Dataset Exploration

Reuses the same `loader` / `profiler` / `explore` modules the web tool uses, so anything you work out here transfers straight into the app.

**Set `DATASET` below** to a filename in `../data`, an absolute path, or an `http(s)` URL.

In [ ]:
import sys, os
from pathlib import Path

# Make the backend package importable from the notebook.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from app import loader, profiler, explore

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

# 'cms:xubh-q36u' pulls Hospital General Information from the CMS catalog (cached to ../data).
DATASET = os.environ.get('DATASET', 'cms:xubh-q36u')  # <-- change me
df = loader.load(DATASET)
print(f'{DATASET}: {len(df):,} rows x {df.shape[1]} cols')
df.head()

## 1. Schema & per-column profile

In [ ]:
prof = profiler.full_profile(df)
print(prof['overview'])

cols = pd.DataFrame([{ 'name': c['name'], 'kind': c['kind'], 'dtype': c['dtype'],
                       'null_pct': c['null_pct'], 'distinct': c['distinct'] }
                     for c in prof['columns']])
cols

## 2. Data-quality inspector

In [ ]:
issues = pd.DataFrame(prof['issues'])
display(issues)

# Null map across columns
null_pct = df.isna().mean().mul(100).sort_values(ascending=False)
if null_pct.max() > 0:
    ax = null_pct[null_pct > 0].plot.barh(figsize=(7, 4), color='#e76f51')
    ax.set_xlabel('% missing'); ax.set_title('Missing values by column'); plt.tight_layout()

## 3. Distributions (numeric columns)

In [ ]:
num_cols = [c['name'] for c in prof['columns'] if c['kind'] == 'numeric']
if num_cols:
    n = len(num_cols)
    fig, axes = plt.subplots((n + 2) // 3, min(n, 3), figsize=(14, 3 * ((n + 2) // 3)))
    for ax, col in zip(list(axes.flat) if n > 1 else [axes], num_cols):
        df[col].dropna().plot.hist(bins=30, ax=ax, color='#0f3460')
        ax.set_title(col)
    plt.tight_layout()
else:
    print('No numeric columns.')

## 4. Group-by aggregation

Same `explore.aggregate` the API uses. Swap `group_by` / `metric` / `agg`.

In [ ]:
cat_cols = [c['name'] for c in prof['columns'] if c['kind'] in ('categorical', 'boolean')]
if cat_cols:
    gb = cat_cols[0]
    metric = num_cols[0] if num_cols else None
    agg = 'sum' if metric else 'count'
    data = explore.aggregate(df, group_by=gb, metric=metric, agg=agg, limit=15)
    agg_df = pd.DataFrame(data)
    display(agg_df)
    agg_df.plot.bar(x='key', y='value', figsize=(9, 4), legend=False,
                    title=f'{agg}({metric or "rows"}) by {gb}', color='#0f3460')
    plt.tight_layout()

## 5. Time series (if a date column exists)

In [ ]:
date_cols = [c['name'] for c in prof['columns'] if c['kind'] == 'datetime']
if date_cols:
    ts = pd.DataFrame(explore.timeseries(df, date_col=date_cols[0], agg='count', freq='M'))
    if not ts.empty:
        ts['period'] = pd.to_datetime(ts['period'])
        ts.plot(x='period', y='value', figsize=(10, 4), legend=False,
                title=f'Rows per month ({date_cols[0]})', color='#2a9d8f')
        plt.tight_layout()
else:
    print('No datetime columns detected.')

## 6. Scratch — dataset-specific work

Build the focused metric / scoring / transform here. When it earns its keep, lift it into `backend/app/` so the web tool gets it too.

In [ ]:
# your analysis here
